# **Project1: Predicting Churn Customers**

## **Dataset Overview**
The dataset is of a telecom comppany that has registered customers on different contracts.
In this project, we are predicting customers that are churned(about to stop using the services) before they actually leave. We are predcting them before they actually stop using the services and later leave.

The dataset has 1000 rows and 8 columns.

The following are the libraries imported to help woth the predictions:
* import sklearn the train_test_split splits the data we need for testing and training.
* from sklearn.linear_model we import LogisticRegression as the algorithm we are going to use to predict the churn results
* metrics are used in determing the accuracy of the model, accuracy_score shows the percentage

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
df_customers = pd.read_csv('telecom_customers.cleaned.csv')

print(len(df_customers))

1000


In [3]:
df_customers

,customer_id,monthly_spend,total_calls,data_usage_gb,customer_service_calls,tenure_months,contract_type,churned
0,1001,771,388,9.4,5,11,Month-to-month,1
1,1002,1072,299,5.3,6,8,Month-to-month,1
2,1003,3437,637,22.0,1,67,Two year,0
3,1004,952,435,4.2,6,6,Month-to-month,1
4,1005,887,407,6.1,9,12,Month-to-month,1
...,...,...,...,...,...,...,...,...
995,1996,2251,608,32.2,1,55,Two year,0
996,1997,1564,475,19.3,1,30,One year,0
997,1998,1578,458,14.7,3,26,One year,0
998,1999,919,217,9.9,4,11,Month-to-month,1


#### **Exploring data**
* We explore the data to make sure it is usable by the model.
* Check for missing values, patterns, and structure of the data

In [4]:
# Explore the data
total_churned = df_customers['churned'].sum()
print(total_churned)

437


In [5]:
print(df_customers.head(10).to_string(index=False))

 customer_id  monthly_spend  total_calls  data_usage_gb  customer_service_calls  tenure_months  contract_type  churned
        1001            771          388            9.4                       5             11 Month-to-month        1
        1002           1072          299            5.3                       6              8 Month-to-month        1
        1003           3437          637           22.0                       1             67       Two year        0
        1004            952          435            4.2                       6              6 Month-to-month        1
        1005            887          407            6.1                       9             12 Month-to-month        1
        1006            943          263            8.2                       8              7 Month-to-month        1
        1007            973          331           12.5                       9              7 Month-to-month        1
        1008           1085          283        

In [6]:
# check for missing values
df_customers.isna().sum()

customer_id               0
monthly_spend             0
total_calls               0
data_usage_gb             0
customer_service_calls    0
tenure_months             0
contract_type             0
churned                   0
dtype: int64

#### **Prepare the data for machine learning**
Here the data is prepared for machine learning, being that machine understand specific patterns we change according to that
* The contract type column data is changed to numerical values
* The customer_id column is dropped because it wont be needed in the predictions
* Features are separated from the targets (The data that is goig to be used for predictions is separated from the data that is being predicted)
* Data is separated from the trainig data and the testing data (trainig data is given 80% of the data and testing data is given 20% of the data)
* 

In [7]:
# Prepare the data for machine learning
# it was converted to numbers
data_encoded = pd.get_dummies(df_customers, columns=['contract_type'], drop_first=False)

In [8]:
# Drop columns that wont be needed
data_encoded = data_encoded.drop(['customer_id'], axis=1)

In [9]:
# separate features and the targets
x = data_encoded.drop('churned', axis=1) #the features used to make predictions(churned)
y = data_encoded['churned'] # the target what we want to predict

In [10]:
# splitting the data into training and test sets
# define variable
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size =0.2,
    random_state = 42, # maintains the same split everytime the code is run
    stratify = y  # maintains the same class portions split as the original dataset
)

#### **Train our ML model**
The LogisticRegression algorithm is used to predict the churned customers
* Used a comparison table to compare the predictions and the actual data of the churned customers.

In [11]:
# Train our ML model
model = LogisticRegression(max_iter=1000, random_state=42) 

In [12]:
model.fit(x_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [13]:
# make predictions
# its given the test data
y_pred = model.predict(x_test) # it returns 010101s, predicted class labels
y_pred_proba = model.predict_proba(x_test)[:, 1] # this returns the actual percentages instead of 0s and 1s, the probability

In [14]:
# get a comparison table to compare the actual churn and the prediction made by the model
comparison = pd.DataFrame({
    'Actual churn': y_test.values[:10],
    'Predicted churn': y_pred[:10],
    'Churn probability': [f"{p*100:.1f}" for p in y_pred_proba[:10]],
})

In [15]:
print(comparison.to_string(index=False))

 Actual churn  Predicted churn Churn probability
            1                1              93.5
            0                0               0.0
            1                1              98.9
            1                1              97.6
            1                1              99.6
            1                1              97.2
            0                0               0.0
            1                1              98.3
            1                1              98.2
            1                1              99.3


In [16]:
# Evaluate model accuracy
accuracy = accuracy_score(y_test, y_pred) # y_test is divided by y_pred

In [17]:
print(f"ACCURACY: {accuracy*100:.2f}%")
# The model is 91.00% accurate about the customers that are churned

ACCURACY: 91.00%
